# Project ROI Lens: Advanced Marketing Attribution & Budget Refinement
**Prepared for**: Chief Marketing Officer, Nexus Consumer Brands  
**Framework Partner**: Ai Palette Insights  
**E-Cell IIT Guwahati Strategy & BI Challenge**

---

## Executive Summary & Solution Paradigm
This notebook contains the complete technical codebase for **Project ROI Lens**. It implements a proprietary marketing intelligence pipeline to transition Nexus from rudimentary "Last-Click" attribution models to a **stochastic, multi-touch Markov Chain attribution model**, combined with a **constrained portfolio optimization** framework to allocate a ₹100 Crore budget next quarter.

The implementation is divided into two distinct phases:
1. **Phase 1: Build the Attribution Engine (Find the Truth)**: Detect and filter fraud/bot traffic from raw logs, compute the state-to-state transition matrix, calculate the Removal Effect Index (REI) for each channel, and classify channels as Primers vs. Closers.
2. **Phase 2: Portfolio Reallocation (Fix the Future)**: Fit ad fatigue (diminishing returns) curves and reallocate ₹10 Crore per brand across 5 channels using a bounded multi-variable solver (`scipy.optimize`).

## Step 1: Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load datasets
df_spend = pd.read_csv('data/campaign_spend.csv')
df_touchpoints = pd.read_csv('data/touchpoints.csv')
df_profiles = pd.read_csv('data/user_profiles.csv')

print(f"Loaded Campaign Spends: {df_spend.shape[0]} rows")
print(f"Loaded Touchpoint Logs: {df_touchpoints.shape[0]} rows")
print(f"Loaded User Profiles: {df_profiles.shape[0]} rows")

## Phase 1: Fraud Auditing (Data Sifter)
We identify and flag bot profiles in the raw logs based on two primary non-human behaviors:
1. **Rapid-fire click bursts**: Consecutive clicks by the same user with a sub-millisecond (< 1ms) time difference.
2. **Zero-dwell clicks**: Clicks of exactly 0.0 seconds without an associated impression on that channel.

In [ ]:
def sift_bot_traffic(df):
    df_sorted = df.copy()
    df_sorted['datetime'] = pd.to_datetime(df_sorted['timestamp'])
    df_sorted = df_sorted.sort_values(by=['user_id', 'datetime']).reset_index(drop=True)
    
    df_sorted['is_bot_event'] = False
    
    # 1. Detect rapid-fire click bursts (< 1ms delta)
    click_mask = df_sorted['event_type'] == 'click'
    df_clicks = df_sorted[click_mask].copy()
    df_clicks['time_diff'] = df_clicks.groupby('user_id')['datetime'].diff().dt.total_seconds()
    
    rapid_clicks = df_clicks['time_diff'] < 0.001
    rapid_indices = df_clicks[rapid_clicks].index
    prev_indices = rapid_indices - 1
    
    valid_prev = [idx for idx in prev_indices if idx in df_clicks.index and df_clicks.loc[idx, 'user_id'] == df_clicks.loc[idx+1, 'user_id']]
    flagged_indices = list(rapid_indices) + list(valid_prev)
    df_sorted.loc[flagged_indices, 'is_bot_event'] = True
    
    # 2. Clicks with 0.0s dwell time and no corresponding impressions
    zero_clicks = df_sorted[(df_sorted['event_type'] == 'click') & (df_sorted['dwell_time'] == 0.0)]
    impressions = df_sorted[df_sorted['event_type'] == 'impression'].groupby(['user_id', 'channel']).size().to_dict()
    
    zero_bot_indices = []
    for idx, row in zero_clicks.iterrows():
        if (row['user_id'], row['channel']) not in impressions:
            zero_bot_indices.append(idx)
    df_sorted.loc[zero_bot_indices, 'is_bot_event'] = True
    
    # Flag all actions of bot users
    bot_users = df_sorted[df_sorted['is_bot_event']]['user_id'].unique()
    df_sorted.loc[df_sorted['user_id'].isin(bot_users), 'is_bot_event'] = True
    
    # Split data
    df_clean = df_sorted[~df_sorted['is_bot_event']].copy()
    df_clean = df_clean.drop(columns=['datetime', 'is_bot_event'])
    
    return df_clean, len(bot_users), df_sorted['is_bot_event'].sum()

df_clean_logs, bot_users_count, bot_events_count = sift_bot_traffic(df_touchpoints)
print(f"Bot Auditing Summary:")
print(f"  - Bot Users Flagged: {bot_users_count} ({bot_users_count/df_profiles.shape[0]*100:.2f}%)")
print(f"  - Bot Events Removed: {bot_events_count}")
print(f"  - Sanitized Logs Remaining: {df_clean_logs.shape[0]} rows")

## Phase 1 (Cont.): Multi-Touch Probabilistic Engine (Markov Chains)
We model user behavior as transitions in a discrete-time Markov chain. The baseline conversion probability from `Start` is computed, and a **Removal Effect Index (REI)** is derived for each channel to determine fractional conversion attribution.

In [ ]:
CHANNELS = ["Instagram", "Google Search", "Influencer Networks", "YouTube", "E-commerce Marketplaces"]
STATES = ["Start"] + CHANNELS + ["Conversion", "Null"]
STATE_TO_IDX = {s: i for i, s in enumerate(STATES)}

def analyze_attribution(df_logs, df_spends, label="Global"):
    # Group by user
    grouped = df_logs.sort_values(by='timestamp').groupby('user_id')
    journeys = []
    conversions_count = 0
    
    for user_id, group in grouped:
        path = []
        converted = False
        for _, row in group.iterrows():
            if row['event_type'] == 'conversion':
                converted = True
            ch = row['channel']
            if not path or path[-1] != ch:
                path.append(ch)
        if converted:
            conversions_count += 1
            journeys.append(["Start"] + path + ["Conversion"])
        else:
            journeys.append(["Start"] + path + ["Null"])
            
    # Transition Probabilities
    transition_counts = np.zeros((len(STATES), len(STATES)))
    for j in journeys:
        for step in range(len(j) - 1):
            transition_counts[STATE_TO_IDX[j[step]], STATE_TO_IDX[j[step+1]]] += 1
            
    P = np.zeros((len(STATES), len(STATES)))
    for i in range(len(STATES)):
        row_sum = transition_counts[i].sum()
        if row_sum > 0:
            P[i] = transition_counts[i] / row_sum
        else:
            P[i, i] = 1.0
            
    # Fundamental matrix: N = (I - Q)^-1
    Q = P[:6, :6]
    R = P[:6, 6:]
    N = np.linalg.inv(np.eye(6) - Q)
    B = N @ R
    base_conv_prob = B[0, 0]
    
    # Removal Effect for each channel
    removal_effects = {}
    for idx, ch in enumerate(CHANNELS, start=1):
        Q_mod = Q.copy()
        R_mod = R.copy()
        Q_mod[idx, :] = 0.0
        R_mod[idx, 0] = 0.0 # Conversion
        R_mod[idx, 1] = 1.0 # Null
        
        N_mod = np.linalg.inv(np.eye(6) - Q_mod)
        B_mod = N_mod @ R_mod
        mod_conv_prob = B_mod[0, 0]
        
        removal_effects[ch] = 1.0 - (mod_conv_prob / base_conv_prob) if base_conv_prob > 0 else 0.0
        
    # Normalize to get REI
    total_re = sum(removal_effects.values())
    rei = {ch: (val/total_re if total_re > 0 else 0.2) for ch, val in removal_effects.items()}
    
    # Attribution and CPA/ROI
    spends = df_spends.groupby('channel')['spend'].sum().to_dict()
    records = []
    for ch in CHANNELS:
        spend = spends.get(ch, 0.0)
        convs = conversions_count * rei[ch]
        cpa = spend / convs if convs > 0 else 0.0
        # Assume ₹20,000 (0.002 Cr) revenue value per conversion
        rev = convs * 0.002
        roi = ((rev - spend) / spend * 100) if spend > 0 else 0.0
        records.append({
            "channel": ch,
            "spend_crore": spend,
            "attributed_conversions": convs,
            "cpa_crore": cpa,
            "roi_pct": roi,
            "rei": rei[ch]
        })
        
    # Funnel Classification (Primer vs Closer)
    classification = {}
    for idx, ch in enumerate(CHANNELS, start=1):
        start_to_c = transition_counts[0, idx]
        total_visits_c = transition_counts[:, idx].sum()
        c_to_conv = transition_counts[idx, 6]
        total_out_c = transition_counts[idx].sum()
        
        primer = start_to_c / total_visits_c if total_visits_c > 0 else 0.0
        closer = c_to_conv / total_out_c if total_out_c > 0 else 0.0
        classification[ch] = {
            "primer_score": primer,
            "closer_score": closer,
            "role": "Primer" if primer > closer else "Closer"
        }
        
    return pd.DataFrame(records), classification, conversions_count

df_attr, funnel_roles, total_convs = analyze_attribution(df_clean_logs, df_spend)
print(f"Global Attribution Report (Conversions: {total_convs}):")
display(df_attr)

## Phase 2: Constrained Budget Optimization (Portfolio Reallocation)
We model channel ad fatigue using a saturating curve: \(f(x) = \alpha \ln(x+1)\).
The alpha parameters are fitted for each channel, and next quarter's ₹10 Crore budget per brand is optimized using a bounded SLSQP solver.

In [ ]:
# Fit alpha parameters from Global results
alphas = {}
for idx, row in df_attr.iterrows():
    ch = row['channel']
    spend = row['spend_crore']
    convs = row['attributed_conversions']
    alphas[ch] = convs / np.log(spend + 1) if spend > 0 else 0.0

budget_cap = 10.0 # 10 Crore per brand
max_channel_cap = 5.0 # Max 5 Crore per channel to ensure diversification

# Objective Function (negative conversions to minimize)
def obj_fun(x):
    val = 0.0
    for i, ch in enumerate(CHANNELS):
        val += alphas[ch] * np.log(x[i] + 1)
    return -val

# Constraint: sum(x) = 10 Crore
cons = {'type': 'eq', 'fun': lambda x: sum(x) - budget_cap}
bounds = [(0.0, max_channel_cap) for _ in range(5)]
x0 = [budget_cap / 5] * 5

res = minimize(obj_fun, x0, method='SLSQP', bounds=bounds, constraints=cons)
opt_spends = res.x
opt_convs = -res.fun

# Compile Comparison
df_comparison = pd.DataFrame({
    "Channel": CHANNELS,
    "Historical Spend": [df_attr[df_attr['channel'] == ch]['spend_crore'].values[0] for ch in CHANNELS],
    "Historical Convs": [df_attr[df_attr['channel'] == ch]['attributed_conversions'].values[0] for ch in CHANNELS],
    "Optimized Spend": opt_spends,
    "Optimized Expected Convs": [alphas[ch] * np.log(opt_spends[i] + 1) for i, ch in enumerate(CHANNELS)]
})

print(f"Portfolio Reallocation Optimization Summary:")
display(df_comparison)
print(f"Historical Expected Conversions (Sum): {df_comparison['Historical Convs'].sum():.2f}")
print(f"Optimized Expected Conversions: {opt_convs:.2f} (+{((opt_convs - df_comparison['Historical Convs'].sum())/df_comparison['Historical Convs'].sum()*100):.2f}%)")

## Visualization: Diminishing Returns Curves

In [ ]:
x_range = np.linspace(0, 10, 100)
for ch in CHANNELS:
    y_values = alphas[ch] * np.log(x_range + 1)
    plt.plot(x_range, y_values, label=f"{ch} (alpha={alphas[ch]:.1f})")

plt.title('Diminishing Returns (Conversions vs Spend) per Channel')
plt.xlabel('Spend (Crore)')
plt.ylabel('Expected Conversions')
plt.legend()
plt.show()